<a href="https://colab.research.google.com/github/venukumarpj/infosys-maritime-ai/blob/main/Milestone1_Auth_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyjwt pyotp nest-asyncio pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 54.0 MB/s eta 0:00:00


In [2]:
import streamlit as st
import jwt
import re
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import sqlite3
from datetime import datetime, timedelta
import os

# ---------------------------------------------------------
# CONFIGURATION & SECRET MANAGEMENT
# ---------------------------------------------------------
# Retrieve secrets from Colab userdata (Fallbacks provided for safe local testing if needed)
try:
    from google.colab import userdata
    JWT_SECRET = userdata.get('JWT_SECRET')
    EMAIL_ADDRESS = userdata.get('EMAIL_ADDRESS')
    EMAIL_PASSWORD = userdata.get('EMAIL_PASSWORD')
except Exception:
    # Fallback or dummy defaults if run outside Colab secrets environment
    JWT_SECRET = os.environ.get("JWT_SECRET", "super_secret_jwt_key_12345")
    EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS", "your_email@gmail.com")
    EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "your_app_password")

ADMIN_USER = "admin"
ADMIN_PASS = "AdminSecure789#"

# ---------------------------------------------------------
# DATABASE SETUP
# ---------------------------------------------------------
def init_db():
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS users (
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password TEXT,
            sec_question TEXT,
            sec_answer TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

def add_user(username, email, password, sec_question, sec_answer):
    try:
        conn = sqlite3.connect('users.db')
        c = conn.cursor()
        c.execute('INSERT INTO users VALUES (?, ?, ?, ?, ?)',
                  (username, email, password, sec_question, sec_answer))
        conn.commit()
        conn.close()
        return True, "Signup successful!"
    except sqlite3.IntegrityError:
        return False, "Username or Email already exists."

def get_user_by_username(username):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT * FROM users WHERE username = ?', (username,))
    user = c.fetchone()
    conn.close()
    return user

def get_user_by_email(email):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT * FROM users WHERE email = ?', (email,))
    user = c.fetchone()
    conn.close()
    return user

def update_password(username_or_email, new_password):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('UPDATE users SET password = ? WHERE username = ? OR email = ?',
              (new_password, username_or_email, username_or_email))
    conn.commit()
    conn.close()

def get_all_users():
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT username, email, sec_question FROM users')
    users = c.fetchall()
    conn.close()
    return users

# ---------------------------------------------------------
# VALIDATION FUNCTIONS (Step 8)
# ---------------------------------------------------------
def validate_email(email):
    # At least 2 letters before @, at least 2 letters between @ and dot, at least 2 letters after final dot
    pattern = r'^[a-zA-Z]{2,}@[a-zA-Z]{2,}\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def validate_password(password):
    if len(password) < 8:
        return False, "Password must be at least 8 characters long."
    if not re.search(r'[A-Z]', password):
        return False, "Password must contain at least one uppercase letter."
    if not re.search(r'[a-z]', password):
        return False, "Password must contain at least one lowercase letter."
    if not re.search(r'\d', password):
        return False, "Password must contain at least one number."
    if not re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        return False, "Password must contain at least one special symbol."
    return True, "Valid password"

# ---------------------------------------------------------
# EMAIL OTP SENDER
# ---------------------------------------------------------
def send_otp_email(recipient_email, otp):
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_ADDRESS
        msg['To'] = recipient_email
        msg['Subject'] = "Password Reset OTP - Infosys Milestone 1"

        body = f"Your One-Time Password (OTP) for password reset is: {otp}\nValid for 10 minutes."
        msg.attach(MIMEText(body, 'plain'))

        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, recipient_email, msg.as_string())
        server.quit()
        return True
    except Exception as e:
        return False

# ---------------------------------------------------------
# JWT SESSION MANAGEMENT (Step 7)
# ---------------------------------------------------------
def create_token(username):
    payload = {
        'sub': username,
        'exp': datetime.utcnow() + timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm='HS256')

def verify_token(token):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=['HS256'])
        return payload['sub']
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

# ---------------------------------------------------------
# STREAMLIT APP UI
# ---------------------------------------------------------
st.set_page_config(page_title="Infosys Milestone 1 - Auth Module", page_icon="🔐", layout="centered")

# Session State Initialization
if 'token' not in st.session_state:
    st.session_state.token = None
if 'is_admin' not in st.session_state:
    st.session_state.is_admin = False
if 'otp_storage' not in st.session_state:
    st.session_state.otp_storage = {}

def main():
    st.sidebar.title("Navigation")

    # Check current auth status
    logged_in_user = verify_token(st.session_state.token) if st.session_state.token else None

    if logged_in_user:
        choice = st.sidebar.radio("Go to", ["Dashboard", "Logout"])
        if choice == "Dashboard":
            user_dashboard(logged_in_user)
        elif choice == "Logout":
            st.session_state.token = None
            st.session_state.is_admin = False
            st.rerun()
    elif st.session_state.is_admin:
        choice = st.sidebar.radio("Go to", ["Admin Dashboard", "Logout"])
        if choice == "Admin Dashboard":
            admin_dashboard()
        elif choice == "Logout":
            st.session_state.is_admin = False
            st.rerun()
    else:
        choice = st.sidebar.radio("Go to", ["Login", "Signup", "Forgot Password", "Admin Login"])
        if choice == "Login":
            login_page()
        elif choice == "Signup":
            signup_page()
        elif choice == "Forgot Password":
            forgot_password_page()
        elif choice == "Admin Login":
            admin_login_page()

# --- LOGIN PAGE (Step 3) ---
def login_page():
    st.title("User Login")
    st.markdown("Enter your credentials to access your dashboard.")

    with st.form("login_form"):
        identifier = st.text_input("Username or Email *")
        password = st.text_input("Password *", type="password")
        submit = st.form_submit_button("Login")

        if submit:
            if not identifier or not password:
                st.error("All mandatory fields must be filled.")
            else:
                user = get_user_by_username(identifier) or get_user_by_email(identifier)
                # Generic error message to prevent account enumeration
                if user and user[2] == password:
                    st.session_state.token = create_token(user[0])
                    st.success("Login successful! Redirecting...")
                    st.rerun()
                else:
                    st.error("Invalid username/email or password.")

# --- SIGNUP PAGE (Step 3 & 8) ---
def signup_page():
    st.title("User Signup")
    st.markdown("Create a new account.")

    with st.form("signup_form"):
        username = st.text_input("Username *")
        email = st.text_input("Email * (Format: ab@cd.ef)")
        password = st.text_input("Password *", type="password")
        confirm_password = st.text_input("Confirm Password *", type="password")

        security_questions = [
            "What was your first pet's name?",
            "What is your mother's maiden name?",
            "What was the name of your first school?",
            "What is your favorite book?"
        ]
        sec_question = st.selectbox("Security Question *", security_questions)
        sec_answer = st.text_input("Security Answer *")

        submit = st.form_submit_button("Signup")

        if submit:
            if not all([username, email, password, confirm_password, sec_answer]):
                st.error("All fields are mandatory.")
            elif not validate_email(email):
                st.error("Invalid email format. Rule: at least 2 letters before @, 2 between @ and dot, 2 after final dot (e.g., ab@cd.ef).")
            else:
                valid_pass, pass_msg = validate_password(password)
                if not valid_pass:
                    st.error(pass_msg)
                elif password != confirm_password:
                    st.error("Passwords do not match.")
                else:
                    success, msg = add_user(username, email, password, sec_question, sec_answer)
                    if success:
                        st.success(msg + " Please proceed to Login.")
                    else:
                        st.error(msg)

# --- FORGOT PASSWORD PAGE (Step 9) ---
def forgot_password_page():
    st.title("Forgot Password Recovery")

    recovery_method = st.radio("Choose Recovery Method", ["Security Question", "Email OTP"])

    if recovery_method == "Security Question":
        with st.form("sec_q_form"):
            username = st.text_input("Enter Username *")
            check_user = st.form_submit_button("Fetch Security Question")

        user = get_user_by_username(username) if username else None

        if user or st.session_state.get('sq_verified', False):
            if user:
                st.session_state['sq_username'] = username
                st.session_state['sq_question'] = user[3]

            with st.form("answer_form"):
                st.write(f"**Security Question:** {st.session_state.get('sq_question', '')}")
                ans = st.text_input("Your Answer *")
                new_pass = st.text_input("New Password *", type="password")
                confirm_pass = st.text_input("Confirm New Password *", type="password")
                reset_btn = st.form_submit_button("Reset Password")

                if reset_btn:
                    user_record = get_user_by_username(st.session_state['sq_username'])
                    if ans.strip().lower() != user_record[4].strip().lower():
                        st.error("Incorrect security answer.")
                    else:
                        valid_p, p_msg = validate_password(new_pass)
                        if not valid_p:
                            st.error(p_msg)
                        elif new_pass != confirm_pass:
                            st.error("Passwords do not match.")
                        else:
                            update_password(st.session_state['sq_username'], new_pass)
                            st.success("Password successfully reset! Please login.")
        elif username and not user:
            st.error("Username not found.")

    elif recovery_method == "Email OTP":
        with st.form("otp_request_form"):
            email = st.text_input("Registered Email *")
            send_btn = st.form_submit_button("Send OTP")

            if send_btn:
                user = get_user_by_email(email)
                if not user:
                    st.error("Email not found in system.")
                else:
                    otp = str(random.randint(100000, 999999))
                    st.session_state.otp_storage[email] = otp
                    if send_otp_email(email, otp):
                        st.success("OTP sent successfully to your email!")
                        st.session_state['otp_email'] = email
                    else:
                        st.error("Failed to send email. Check SMTP server/App password settings.")

        if 'otp_email' in st.session_state:
            with st.form("otp_verify_form"):
                entered_otp = st.text_input("Enter 6-digit OTP *")
                new_p = st.text_input("New Password *", type="password")
                conf_p = st.text_input("Confirm New Password *", type="password")
                verify_btn = st.form_submit_button("Verify & Reset Password")

                if verify_btn:
                    target_email = st.session_state['otp_email']
                    if st.session_state.otp_storage.get(target_email) == entered_otp:
                        valid_p, p_msg = validate_password(new_p)
                        if not valid_p:
                            st.error(p_msg)
                        elif new_p != conf_p:
                            st.error("Passwords do not match.")
                        else:
                            update_password(target_email, new_p)
                            st.success("Password reset successful! Please login.")
                            del st.session_state.otp_storage[target_email]
                            del st.session_state['otp_email']
                    else:
                        st.error("Invalid or expired OTP.")

# --- ADMIN LOGIN & DASHBOARD (Step 11) ---
def admin_login_page():
    st.title("Admin Login")
    with st.form("admin_form"):
        user = st.text_input("Admin Username *")
        pwd = st.text_input("Admin Password *", type="password")
        sub = st.form_submit_button("Login as Admin")

        if sub:
            if user == ADMIN_USER and pwd == ADMIN_PASS:
                st.session_state.is_admin = True
                st.success("Admin login successful!")
                st.rerun()
            else:
                st.error("Invalid admin credentials.")

def admin_dashboard():
    st.title("Admin Dashboard")
    st.markdown("### Registered Users List")
    users = get_all_users()
    if users:
        user_data = [{"Username": u[0], "Email": u[1], "Security Question": u[2]} for u in users]
        st.table(user_data)
    else:
        st.info("No registered users found.")

# --- USER DASHBOARD (Step 10) ---
def user_dashboard(username):
    st.title("User Dashboard")
    st.success(f"Welcome back, **{username}**!")
    st.markdown("You have successfully logged in using your secure JWT session.")

    st.info("Explore your secure user workspace features here.")

if __name__ == '__main__':
    main()

2026-10-18 14:37:02.924 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-18 14:37:02.925 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-18 14:37:02.926 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-10-18 14:37:02.928 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-18 14:37:02.929 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-18 14:37:02.931 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

In [8]:
from pyngrok import ngrok
from google.colab import userdata
import os

# Kill any existing ngrok tunnels
ngrok.kill()

# Retrieve ngrok token securely from Colab secrets
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Run streamlit in the background
get_ipython().system_raw("streamlit run app.py &")

# Open public ngrok tunnel for port 8501
public_url = ngrok.connect(8501)
print(f" * Public URL: {public_url}")

 * Public URL: NgrokTunnel: "https://quail-shrubbery-wolf.ngrok-free.dev" -> "http://localhost:8501"


In [7]:
%%writefile app.py
import streamlit as st
import jwt
import re
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import sqlite3
from datetime import datetime, timedelta
import os

# Retrieve secrets from Colab userdata
try:
    from google.colab import userdata
    JWT_SECRET = userdata.get('JWT_SECRET')
    EMAIL_ADDRESS = userdata.get('EMAIL_ADDRESS')
    EMAIL_PASSWORD = userdata.get('EMAIL_PASSWORD')
except Exception:
    JWT_SECRET = os.environ.get("JWT_SECRET", "super_secret_jwt_key_12345")
    EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS", "your_email@gmail.com")
    EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "your_app_password")

ADMIN_USER = "admin"
ADMIN_PASS = "AdminSecure789#"

def init_db():
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS users (
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password TEXT,
            sec_question TEXT,
            sec_answer TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

def add_user(username, email, password, sec_question, sec_answer):
    try:
        conn = sqlite3.connect('users.db')
        c = conn.cursor()
        c.execute('INSERT INTO users VALUES (?, ?, ?, ?, ?)',
                  (username, email, password, sec_question, sec_answer))
        conn.commit()
        conn.close()
        return True, "Signup successful!"
    except sqlite3.IntegrityError:
        return False, "Username or Email already exists."

def get_user_by_username(username):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT * FROM users WHERE username = ?', (username,))
    user = c.fetchone()
    conn.close()
    return user

def get_user_by_email(email):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT * FROM users WHERE email = ?', (email,))
    user = c.fetchone()
    conn.close()
    return user

def update_password(username_or_email, new_password):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('UPDATE users SET password = ? WHERE username = ? OR email = ?',
              (new_password, username_or_email, username_or_email))
    conn.commit()
    conn.close()

def get_all_users():
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT username, email, sec_question FROM users')
    users = c.fetchall()
    conn.close()
    return users

def validate_email(email):
    pattern = r'^[a-zA-Z]{2,}@[a-zA-Z]{2,}\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def validate_password(password):
    if len(password) < 8:
        return False, "Password must be at least 8 characters long."
    if not re.search(r'[A-Z]', password):
        return False, "Password must contain at least one uppercase letter."
    if not re.search(r'[a-z]', password):
        return False, "Password must contain at least one lowercase letter."
    if not re.search(r'\d', password):
        return False, "Password must contain at least one number."
    if not re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        return False, "Password must contain at least one special symbol."
    return True, "Valid password"

def send_otp_email(recipient_email, otp):
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_ADDRESS
        msg['To'] = recipient_email
        msg['Subject'] = "Password Reset OTP - Infosys Milestone 1"
        body = f"Your One-Time Password (OTP) for password reset is: {otp}\nValid for 10 minutes."
        msg.attach(MIMEText(body, 'plain'))
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, recipient_email, msg.as_string())
        server.quit()
        return True
    except Exception as e:
        return False

def create_token(username):
    payload = {
        'sub': username,
        'exp': datetime.utcnow() + timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm='HS256')

def verify_token(token):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=['HS256'])
        return payload['sub']
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

st.set_page_config(page_title="Infosys Milestone 1 - Auth Module", page_icon="🔐", layout="centered")

if 'token' not in st.session_state:
    st.session_state.token = None
if 'is_admin' not in st.session_state:
    st.session_state.is_admin = False
if 'otp_storage' not in st.session_state:
    st.session_state.otp_storage = {}

def main():
    st.sidebar.title("Navigation")
    logged_in_user = verify_token(st.session_state.token) if st.session_state.token else None

    if logged_in_user:
        choice = st.sidebar.radio("Go to", ["Dashboard", "Logout"])
        if choice == "Dashboard":
            user_dashboard(logged_in_user)
        elif choice == "Logout":
            st.session_state.token = None
            st.session_state.is_admin = False
            st.rerun()
    elif st.session_state.is_admin:
        choice = st.sidebar.radio("Go to", ["Admin Dashboard", "Logout"])
        if choice == "Admin Dashboard":
            admin_dashboard()
        elif choice == "Logout":
            st.session_state.is_admin = False
            st.rerun()
    else:
        choice = st.sidebar.radio("Go to", ["Login", "Signup", "Forgot Password", "Admin Login"])
        if choice == "Login":
            login_page()
        elif choice == "Signup":
            signup_page()
        elif choice == "Forgot Password":
            forgot_password_page()
        elif choice == "Admin Login":
            admin_login_page()

def login_page():
    st.title("User Login")
    st.markdown("Enter your credentials to access your dashboard.")
    with st.form("login_form"):
        identifier = st.text_input("Username or Email *")
        password = st.text_input("Password *", type="password")
        submit = st.form_submit_button("Login")
        if submit:
            if not identifier or not password:
                st.error("All mandatory fields must be filled.")
            else:
                user = get_user_by_username(identifier) or get_user_by_email(identifier)
                if user and user[2] == password:
                    st.session_state.token = create_token(user[0])
                    st.success("Login successful! Redirecting...")
                    st.rerun()
                else:
                    st.error("Invalid username/email or password.")

def signup_page():
    st.title("User Signup")
    st.markdown("Create a new account.")
    with st.form("signup_form"):
        username = st.text_input("Username *")
        email = st.text_input("Email * (Format: ab@cd.ef)")
        password = st.text_input("Password *", type="password")
        confirm_password = st.text_input("Confirm Password *", type="password")
        security_questions = [
            "What was your first pet's name?",
            "What is your mother's maiden name?",
            "What was the name of your first school?",
            "What is your favorite book?"
        ]
        sec_question = st.selectbox("Security Question *", security_questions)
        sec_answer = st.text_input("Security Answer *")
        submit = st.form_submit_button("Signup")
        if submit:
            if not all([username, email, password, confirm_password, sec_answer]):
                st.error("All fields are mandatory.")
            elif not validate_email(email):
                st.error("Invalid email format. Rule: at least 2 letters before @, 2 between @ and dot, 2 after final dot (e.g., ab@cd.ef).")
            else:
                valid_pass, pass_msg = validate_password(password)
                if not valid_pass:
                    st.error(pass_msg)
                elif password != confirm_password:
                    st.error("Passwords do not match.")
                else:
                    success, msg = add_user(username, email, password, sec_question, sec_answer)
                    if success:
                        st.success(msg + " Please proceed to Login.")
                    else:
                        st.error(msg)

def forgot_password_page():
    st.title("Forgot Password Recovery")
    recovery_method = st.radio("Choose Recovery Method", ["Security Question", "Email OTP"])
    if recovery_method == "Security Question":
        with st.form("sec_q_form"):
            username = st.text_input("Enter Username *")
            check_user = st.form_submit_button("Fetch Security Question")
        user = get_user_by_username(username) if username else None
        if user or st.session_state.get('sq_verified', False):
            if user:
                st.session_state['sq_username'] = username
                st.session_state['sq_question'] = user[3]
            with st.form("answer_form"):
                st.write(f"**Security Question:** {st.session_state.get('sq_question', '')}")
                ans = st.text_input("Your Answer *")
                new_pass = st.text_input("New Password *", type="password")
                confirm_pass = st.text_input("Confirm New Password *", type="password")
                reset_btn = st.form_submit_button("Reset Password")
                if reset_btn:
                    user_record = get_user_by_username(st.session_state['sq_username'])
                    if ans.strip().lower() != user_record[4].strip().lower():
                        st.error("Incorrect security answer.")
                    else:
                        valid_p, p_msg = validate_password(new_pass)
                        if not valid_p:
                            st.error(p_msg)
                        elif new_pass != confirm_pass:
                            st.error("Passwords do not match.")
                        else:
                            update_password(st.session_state['sq_username'], new_pass)
                            st.success("Password successfully reset! Please login.")
        elif username and not user:
            st.error("Username not found.")
    elif recovery_method == "Email OTP":
        with st.form("otp_request_form"):
            email = st.text_input("Registered Email *")
            send_btn = st.form_submit_button("Send OTP")
            if send_btn:
                user = get_user_by_email(email)
                if not user:
                    st.error("Email not found in system.")
                else:
                    otp = str(random.randint(100000, 999999))
                    st.session_state.otp_storage[email] = otp
                    if send_otp_email(email, otp):
                        st.success("OTP sent successfully to your email!")
                        st.session_state['otp_email'] = email
                    else:
                        st.error("Failed to send email. Check SMTP server/App password settings.")
        if 'otp_email' in st.session_state:
            with st.form("otp_verify_form"):
                entered_otp = st.text_input("Enter 6-digit OTP *")
                new_p = st.text_input("New Password *", type="password")
                conf_p = st.text_input("Confirm New Password *", type="password")
                verify_btn = st.form_submit_button("Verify & Reset Password")
                if verify_btn:
                    target_email = st.session_state['otp_email']
                    if st.session_state.otp_storage.get(target_email) == entered_otp:
                        valid_p, p_msg = validate_password(new_p)
                        if not valid_p:
                            st.error(p_msg)
                        elif new_p != conf_p:
                            st.error("Passwords do not match.")
                        else:
                            update_password(target_email, new_p)
                            st.success("Password reset successful! Please login.")
                            del st.session_state.otp_storage[target_email]
                            del st.session_state['otp_email']
                    else:
                        st.error("Invalid or expired OTP.")

def admin_login_page():
    st.title("Admin Login")
    with st.form("admin_form"):
        user = st.text_input("Admin Username *")
        pwd = st.text_input("Admin Password *", type="password")
        sub = st.form_submit_button("Login as Admin")
        if sub:
            if user == ADMIN_USER and pwd == ADMIN_PASS:
                st.session_state.is_admin = True
                st.success("Admin login successful!")
                st.rerun()
            else:
                st.error("Invalid admin credentials.")

def admin_dashboard():
    st.title("Admin Dashboard")
    st.markdown("### Registered Users List")
    users = get_all_users()
    if users:
        user_data = [{"Username": u[0], "Email": u[1], "Security Question": u[2]} for u in users]
        st.table(user_data)
    else:
        st.info("No registered users found.")

def user_dashboard(username):
    st.title("User Dashboard")
    st.success(f"Welcome back, **{username}**!")
    st.markdown("You have successfully logged in using your secure JWT session.")

if __name__ == '__main__':
    main()

Writing app.py
